# B2S 01 - AndinaLog Productos

Conversión Bronze a Silver y cuarentena de productos logísticos. La fuente Bronze se conserva sin modificaciones; todas las decisiones, transformaciones, imputaciones y errores residuales quedan auditados.

In [1]:
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)


def find_root():
    candidates = []
    if os.getenv("ANDINALOG_ROOT"):
        candidates.append(Path(os.environ["ANDINALOG_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "datos" / "bronze").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró el directorio datos/bronze")


ROOT = find_root()
EXECUTED_AT_UTC = datetime.now(timezone.utc).isoformat()
CONFIG = {
    "entidad": "producto_logistico",
    "granularidad": "una fila por producto_id normalizado",
    "clave": "producto_id",
    "rutas": {
        "bronze": "datos/bronze/andinalog_productos.csv",
        "silver": "datos/silver/andinalog_productos_silver.csv",
        "quarantine": "datos/quarantine/andinalog_productos_quarantine.csv",
        "informe": "informes/bronze_silver/Informe_B2S_01_Productos.md",
        "notebook": "notebooks/bronze_silver/01_productos/B2S_01_AndinaLog_Productos.ipynb",
    },
    "lectura": {"encoding": "utf-8", "dtype": "str", "keep_default_na": False},
    "columnas": {
        "texto": ["producto_id", "nombre_producto", "categoria_logistica"],
        "numericas": [
            "temperatura_conservacion_requerida_c",
            "tolerancia_temperatura_c",
            "precio_unitario_bob",
            "costo_unitario_bob",
        ],
        "obligatorias": [
            "producto_id", "nombre_producto", "categoria_logistica",
            "temperatura_conservacion_requerida_c", "tolerancia_temperatura_c",
            "precio_unitario_bob", "costo_unitario_bob",
        ],
        "permitir_extra": False,
    },
    "formatos": {"producto_id": r"^PROD-\d{3}$"},
    "unidad_temperatura": {"canon": "Celsius", "fahrenheit_convertible": True, "kelvin_esperado": False},
    "zonas_horarias": {"aplica_a_esta_fuente": False, "sin_zona": "America/La_Paz", "silver": "UTC"},
    "categorias_validas": ["Fresco", "Seco", "Congelado"],
    "alias_categoria": {"conjelado": "Congelado"},
    "temperatura_por_categoria": {"Fresco": 4.0, "Seco": 20.0, "Congelado": -18.0},
    "tolerancia_por_categoria": {"Fresco": 2.0, "Seco": 5.0, "Congelado": 2.0},
    "evidencia_catalogo": "correspondencia consistente en filas con categorias canonicas; alias conflictivos se conservan en cuarentena",
    "rangos": {"tolerancia_min": 0.0, "precio_min": 0.0, "costo_min": 0.0},
    "centinelas": [-999],
    "imputaciones": {
        "temperatura_por_categoria": {
            "habilitada": True,
            "metodo": "catalogo_deterministico_categoria",
            "motivo": "categoria canonica con correspondencia unica declarada",
        }
    },
    "politica_duplicados": {
        "metodo": "cuarentena_de_todas_las_ocurrencias",
        "motivo": "no elegir arbitrariamente una ocurrencia de una clave funcional duplicada",
    },
}
PATHS = {name: ROOT / relative for name, relative in CONFIG["rutas"].items()}
print("Raíz detectada:", ROOT)
print("Configuración centralizada para:", CONFIG["entidad"])
print("Fecha de ejecución UTC:", EXECUTED_AT_UTC)


Raíz detectada: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Configuración centralizada para: producto_logistico
Fecha de ejecución UTC: 2026-09-24T17:35:48.441929+00:00


In [2]:
bronze = pd.read_csv(PATHS["bronze"], **CONFIG["lectura"])
perfil = {
    "filas": len(bronze),
    "columnas": bronze.columns.tolist(),
    "tipos_recibidos": bronze.dtypes.astype(str).to_dict(),
    "nulos_o_vacios": bronze.eq("").sum().to_dict(),
    "duplicados_clave_normalizada": int(bronze["producto_id"].str.strip().str.upper().duplicated(keep=False).sum()),
    "categorias": bronze["categoria_logistica"].value_counts(dropna=False).to_dict(),
}
print("Perfil Bronze")
print("Filas:", perfil["filas"])
print("Tipos recibidos:", perfil["tipos_recibidos"])
print("Nulos o vacíos:", perfil["nulos_o_vacios"])
print("Filas con clave duplicada normalizada:", perfil["duplicados_clave_normalizada"])
print("Categorías:", perfil["categorias"])


Perfil Bronze
Filas: 63
Tipos recibidos: {'producto_id': 'str', 'nombre_producto': 'str', 'categoria_logistica': 'str', 'temperatura_conservacion_requerida_c': 'str', 'tolerancia_temperatura_c': 'str', 'precio_unitario_bob': 'str', 'costo_unitario_bob': 'str'}
Nulos o vacíos: {'producto_id': 0, 'nombre_producto': 0, 'categoria_logistica': 0, 'temperatura_conservacion_requerida_c': 2, 'tolerancia_temperatura_c': 0, 'precio_unitario_bob': 0, 'costo_unitario_bob': 0}
Filas con clave duplicada normalizada: 6
Categorías: {'Seco': 32, 'Fresco': 18, 'Congelado': 10, 'conjelado': 3}


In [3]:
def append_reason(df, mask, column, reason):
    df.loc[mask, column] = df.loc[mask, column].map(
        lambda current: reason if not current else f"{current} | {reason}"
    )
    return df


def validar_contrato_entrada(df):
    df = df.copy()
    expected = set(CONFIG["columnas"]["obligatorias"])
    received = set(df.columns)
    missing = sorted(expected - received)
    extra = sorted(received - expected)
    if missing or (extra and not CONFIG["columnas"]["permitir_extra"]):
        raise ValueError(f"Contrato de entrada inválido; faltantes={missing}, extra={extra}")
    return df


def estructurar(df):
    df = df.copy()
    df["_fila_bronze"] = range(2, len(df) + 2)
    for column in ["errores_bloqueantes", "motivos_transformacion", "motivos_imputacion"]:
        df[column] = ""
    for column in CONFIG["columnas"]["texto"] + CONFIG["columnas"]["numericas"]:
        df[f"{column}_original"] = df[column]
    return df


def normalizar(df):
    df = df.copy()
    df["producto_id_tratado"] = df["producto_id"].str.strip().str.upper()
    df["nombre_producto_tratado"] = df["nombre_producto"].str.strip()
    df["categoria_logistica_tratado"] = df["categoria_logistica"].str.strip()
    for alias, canonica in CONFIG["alias_categoria"].items():
        mask = df["categoria_logistica_tratado"].str.casefold().eq(alias.casefold())
        df.loc[mask, "categoria_logistica_tratado"] = canonica
    for column in CONFIG["columnas"]["texto"]:
        df[f"{column}_transformado"] = df[column].ne(df[f"{column}_tratado"])
    append_reason(df, df["producto_id_transformado"], "motivos_transformacion", "identificador_normalizado")
    append_reason(df, df["nombre_producto_transformado"], "motivos_transformacion", "nombre_normalizado")
    append_reason(df, df["categoria_logistica_transformado"], "motivos_transformacion", "alias_categoria")
    return df


def convertir_numericos(df):
    df = df.copy()
    for column in CONFIG["columnas"]["numericas"]:
        treated = f"{column}_tratado"
        invalid_flag = f"{column}_conversion_invalida"
        sentinel_flag = f"{column}_centinela_detectado"
        df[treated] = pd.to_numeric(df[column], errors="coerce")
        df[invalid_flag] = df[treated].isna() & df[column].ne("")
        append_reason(df, df[invalid_flag], "errores_bloqueantes", f"conversion_invalida:{column}")
        df[sentinel_flag] = df[treated].isin(CONFIG["centinelas"])
        df.loc[df[sentinel_flag], treated] = pd.NA
        append_reason(df, df[sentinel_flag], "errores_bloqueantes", f"centinela:{column}")
    return df


def validar_pre_imputacion(df):
    df = df.copy()
    key = "producto_id_tratado"
    invalid_key = ~df[key].fillna("").str.match(CONFIG["formatos"]["producto_id"])
    duplicate = df[key].duplicated(keep=False) & df[key].notna()
    invalid_category = ~df["categoria_logistica_tratado"].isin(CONFIG["categorias_validas"])
    append_reason(df, invalid_key, "errores_bloqueantes", "producto_id_invalido")
    append_reason(df, duplicate, "errores_bloqueantes", "producto_id_duplicado")
    append_reason(df, invalid_category, "errores_bloqueantes", "categoria_invalida")
    return df


def imputar_temperatura(df):
    df = df.copy()
    temperature = "temperatura_conservacion_requerida_c_tratado"
    category = "categoria_logistica_tratado"
    rule = CONFIG["imputaciones"]["temperatura_por_categoria"]
    mask = rule["habilitada"] & df[temperature].isna() & df[category].isin(CONFIG["categorias_validas"])
    df.loc[mask, temperature] = df.loc[mask, category].map(CONFIG["temperatura_por_categoria"])
    df["temperatura_conservacion_requerida_c_imputada"] = mask
    df["temperatura_conservacion_requerida_c_imputacion_metodo"] = ""
    df.loc[mask, "temperatura_conservacion_requerida_c_imputacion_metodo"] = rule["metodo"]
    df["imputacion_metodo"] = ""
    df.loc[mask, "imputacion_metodo"] = rule["metodo"]
    df["imputacion_motivo"] = ""
    df.loc[mask, "imputacion_motivo"] = rule["motivo"]
    append_reason(df, mask, "motivos_imputacion", rule["motivo"])
    return df


def validar_post_tratamiento(df):
    df = df.copy()
    for column in CONFIG["columnas"]["obligatorias"]:
        missing = df[f"{column}_tratado"].eq("") if column in CONFIG["columnas"]["texto"] else df[f"{column}_tratado"].isna()
        append_reason(df, missing, "errores_bloqueantes", f"requerido_ausente:{column}")
    category = "categoria_logistica_tratado"
    temperature = "temperatura_conservacion_requerida_c_tratado"
    tolerance = "tolerancia_temperatura_c_tratado"
    price = "precio_unitario_bob_tratado"
    cost = "costo_unitario_bob_tratado"
    expected_temperature = df[category].map(CONFIG["temperatura_por_categoria"])
    expected_tolerance = df[category].map(CONFIG["tolerancia_por_categoria"])
    append_reason(df, df[temperature].notna() & expected_temperature.notna() & df[temperature].ne(expected_temperature), "errores_bloqueantes", "temperatura_categoria_incompatible")
    append_reason(df, df[tolerance].notna() & expected_tolerance.notna() & df[tolerance].ne(expected_tolerance), "errores_bloqueantes", "tolerancia_categoria_incompatible")
    append_reason(df, df[tolerance].notna() & df[tolerance].lt(CONFIG["rangos"]["tolerancia_min"]), "errores_bloqueantes", "tolerancia_fuera_de_rango")
    append_reason(df, df[price].notna() & df[price].lt(CONFIG["rangos"]["precio_min"]), "errores_bloqueantes", "precio_fuera_de_rango")
    append_reason(df, df[cost].notna() & df[cost].lt(CONFIG["rangos"]["costo_min"]), "errores_bloqueantes", "costo_fuera_de_rango")
    append_reason(df, df[price].notna() & df[cost].notna() & df[cost].gt(df[price]), "errores_bloqueantes", "costo_mayor_precio")
    return df


def asignar_calidad(df):
    df = df.copy()
    transform_flags = [f"{column}_transformado" for column in CONFIG["columnas"]["texto"]]
    df["fue_transformada"] = df[transform_flags].any(axis=1)
    df["fue_imputada"] = df["temperatura_conservacion_requerida_c_imputada"]
    df["conteo_transformaciones"] = df[transform_flags].sum(axis=1).astype(int)
    df["conteo_imputaciones"] = df["fue_imputada"].astype(int)
    df["calidad_motivo"] = df["errores_bloqueantes"].mask(df["errores_bloqueantes"].eq(""), "sin_incidencias")
    df["calidad_estado"] = "valida"
    df.loc[df["fue_transformada"], "calidad_estado"] = "valida_con_transformacion"
    df.loc[df["fue_imputada"], "calidad_estado"] = "valida_con_imputacion"
    df.loc[df["fue_transformada"] & df["fue_imputada"], "calidad_estado"] = "valida_con_transformacion_e_imputacion"
    df.loc[df["errores_bloqueantes"].ne(""), "calidad_estado"] = "cuarentena"
    return df


work = (
    bronze.pipe(validar_contrato_entrada)
    .pipe(estructurar)
    .pipe(normalizar)
    .pipe(convertir_numericos)
    .pipe(validar_pre_imputacion)
    .pipe(imputar_temperatura)
    .pipe(validar_post_tratamiento)
    .pipe(asignar_calidad)
)
silver = work.loc[work["calidad_estado"].ne("cuarentena")].copy()
quarantine = work.loc[work["calidad_estado"].eq("cuarentena")].copy()
for column in CONFIG["columnas"]["texto"] + CONFIG["columnas"]["numericas"]:
    silver[column] = silver[f"{column}_tratado"]
    quarantine[column] = quarantine[f"{column}_tratado"]
silver.to_csv(PATHS["silver"], index=False, encoding="utf-8")
quarantine.to_csv(PATHS["quarantine"], index=False, encoding="utf-8")
print({"bronze": len(bronze), "silver": len(silver), "quarantine": len(quarantine)})
print("Estados Silver:", silver["calidad_estado"].value_counts().to_dict())
print("Motivos cuarentena:", quarantine["errores_bloqueantes"].value_counts().to_dict())


{'bronze': 63, 'silver': 54, 'quarantine': 9}
Estados Silver: {'valida': 47, 'valida_con_transformacion': 5, 'valida_con_imputacion': 2}
Motivos cuarentena: {'producto_id_duplicado': 6, 'temperatura_categoria_incompatible': 2, 'temperatura_categoria_incompatible | tolerancia_categoria_incompatible': 1}


In [4]:
silver_file = pd.read_csv(PATHS["silver"])
quarantine_file = pd.read_csv(PATHS["quarantine"])
assert len(bronze) == len(silver_file) + len(quarantine_file)
assert set(silver_file["_fila_bronze"]).isdisjoint(set(quarantine_file["_fila_bronze"]))
assert set(silver_file["_fila_bronze"]) | set(quarantine_file["_fila_bronze"]) == set(work["_fila_bronze"])
assert silver_file["producto_id"].is_unique
assert silver_file["errores_bloqueantes"].fillna("").eq("").all()
assert set(silver_file["categoria_logistica"]).issubset(CONFIG["categorias_validas"])
assert silver_file["temperatura_conservacion_requerida_c"].notna().all()
assert silver_file["tolerancia_temperatura_c"].notna().all()
assert set(["temperatura_conservacion_requerida_c_original", "temperatura_conservacion_requerida_c_tratado", "temperatura_conservacion_requerida_c_imputada", "temperatura_conservacion_requerida_c_imputacion_metodo", "imputacion_metodo", "imputacion_motivo", "motivos_imputacion", "fue_imputada", "calidad_estado"]).issubset(silver_file.columns)
imputed_file = silver_file.loc[silver_file["fue_imputada"]]
assert imputed_file["temperatura_conservacion_requerida_c_imputacion_metodo"].eq(CONFIG["imputaciones"]["temperatura_por_categoria"]["metodo"]).all()
assert imputed_file["imputacion_metodo"].eq(CONFIG["imputaciones"]["temperatura_por_categoria"]["metodo"]).all()
assert imputed_file["imputacion_motivo"].eq(CONFIG["imputaciones"]["temperatura_por_categoria"]["motivo"]).all()
assert imputed_file["motivos_imputacion"].fillna("").ne("").all()
assert silver_file.loc[~silver_file["fue_imputada"], "temperatura_conservacion_requerida_c_imputacion_metodo"].fillna("").eq("").all()

quality = silver_file["calidad_estado"].value_counts().to_dict()
reasons = quarantine_file["errores_bloqueantes"].value_counts().to_dict()
report = [
    "# Informe B2S 01 - AndinaLog Productos", "",
    "## Objetivo, entidad y granularidad",
    "Conversión auditada de productos logísticos desde Bronze a Silver y cuarentena.",
    f"- Entidad: {CONFIG['entidad']}.",
    f"- Granularidad: {CONFIG['granularidad']}.",
    "- Clave funcional: `producto_id` normalizado.",
    "",
    "## Contrato de entrada",
    f"- Columnas obligatorias: {CONFIG['columnas']['obligatorias']}.",
    f"- Lectura Bronze: {CONFIG['lectura']}; tipos tratados: texto para identificador/nombre/categoria y numerico para temperatura, tolerancia, precio y costo.",
    f"- El esquema exige exactamente las columnas configuradas; columnas extra permitidas: {CONFIG['columnas']['permitir_extra']}.",
    "",
    "## Perfil Bronze de esta ejecución",
    f"- Filas Bronze: {perfil['filas']}.",
    f"- Columnas: {perfil['columnas']}.",
    f"- Tipos recibidos: {perfil['tipos_recibidos']}.",
    f"- Valores vacíos: {perfil['nulos_o_vacios']}.",
    f"- Filas con clave duplicada normalizada: {perfil['duplicados_clave_normalizada']}.",
    f"- Categorías observadas: {perfil['categorias']}.",
    "",
    "## Reglas, transformaciones e imputación",
    "- Celsius es la unidad canónica; no hay columnas de fecha ni unidades Fahrenheit/Kelvin en esta fuente.",
    "- Se recortan espacios y se normaliza el identificador a mayúsculas.",
    "- La representación `conjelado` se corrige determinísticamente a `Congelado` y conserva auditoría.",
    "- Cada conversión numérica conserva original, tratado, bandera de conversión inválida y bandera de centinela.",
    f"- La temperatura vacía se imputa exclusivamente con el método `{CONFIG['imputaciones']['temperatura_por_categoria']['metodo']}` declarado en `CONFIG`; conserva original, tratado, bandera, método y motivo tanto por campo como a nivel de fila, sin estadísticas ni información futura.",
    "- Se valida la temperatura y la tolerancia contra el catálogo de la categoría, además de rangos monetarios y `costo <= precio`.",
    "",
    "## Enrutamiento y controles",
    f"- Política de duplicados: {CONFIG['politica_duplicados']['metodo']} ({CONFIG['politica_duplicados']['motivo']}).",
    f"- Silver: {len(silver_file)} filas; estados: {quality}.",
    f"- Filas con imputación: {int(silver_file['fue_imputada'].sum())}.",
    f"- Cuarentena: {len(quarantine_file)} filas; motivos: {reasons}.",
    f"- Conciliación persistida: Bronze {len(bronze)} = Silver {len(silver_file)} + cuarentena {len(quarantine_file)}.",
    "- Silver no contiene errores bloqueantes y su clave es única.",
    "- Las filas persistidas de Silver y cuarentena no se solapan y su unión cubre todas las filas Bronze.",
    "",
    "## Limitaciones y decisiones defendibles",
    f"- El catálogo categoría-temperatura y categoría-tolerancia se deriva de {CONFIG['evidencia_catalogo']} y se declara explícitamente en `CONFIG`; requiere confirmación si aparece un contrato maestro posterior.",
    "- No se elige una ocurrencia de claves duplicadas, porque la fuente no aporta una regla de precedencia reproducible.",
    "- Las incompatibilidades entre categoría normalizada y temperatura se conservan en cuarentena para corrección operativa.",
    "",
    "## Archivos generados",
    f"- `{CONFIG['rutas']['notebook']}`",
    f"- `{CONFIG['rutas']['silver']}`",
    f"- `{CONFIG['rutas']['quarantine']}`",
    f"- `{CONFIG['rutas']['informe']}`",
    "",
    "## Reproducibilidad",
    f"- Fecha de ejecución UTC: {EXECUTED_AT_UTC}.",
    f"- Python: {platform.python_version()}.",
    f"- pandas: {pd.__version__}.",
    "- Bronze se lee con `dtype=str` y no se modifica.",
    "- La raíz se detecta mediante `ANDINALOG_ROOT` o buscando el directorio `datos/bronze`; la fuente se resuelve desde `CONFIG['rutas']['bronze']`.",
    f"- Zona de fechas sin zona: {CONFIG['zonas_horarias']['sin_zona']}; destino Silver: {CONFIG['zonas_horarias']['silver']}; no aplica por ausencia de fechas.",
    f"- Conteos: Bronze {len(bronze)}, Silver {len(silver_file)}, cuarentena {len(quarantine_file)}.",
    "- Ejecutar las celdas en orden; los controles finales se realizan sobre los CSV ya persistidos.",
]
PATHS["informe"].write_text("\n".join(report) + "\n", encoding="utf-8")
print("Controles persistidos: OK")
print(pd.DataFrame({"bronze": [len(bronze)], "silver": [len(silver_file)], "quarantine": [len(quarantine_file)]}))
print("Informe generado:", PATHS["informe"])


Controles persistidos: OK
   bronze  silver  quarantine
0      63      54           9
Informe generado: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\bronze_silver\Informe_B2S_01_Productos.md
